In [1]:
import kaleidocell
import scanpy as sc
import os


In [2]:
adata = sc.read_h5ad("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/01_extGBmap/immune_subsets/Neutrophil.h5ad")


In [4]:
print(adata.obs.columns.tolist())
print(adata.obs["donor_id"].nunique())
print(f"{adata.obs['annotation_level_3'].unique()[0]}: {adata.n_obs} cells")

['author', 'donor_id', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'sex_ontology_term_id', 'annotation_level_1', 'annotation_level_2', 'annotation_level_3', 'gbmap', 'suspension_type', 'stage', 'location', 'sector', 'celltype_original', 'EGFR', 'MET', 'p53', 'TERT', 'ATRX', 'PTEN', 'MGMT', 'chr1p19q', 'PDGFR', 'tissue_ontology_term_id', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid']
72
Neutrophil: 5117 cells


In [5]:
results_nmf, nmf_convergence = kaleidocell.multi_sample_nmf(
    adata,
    batch_key="donor_id",
    test_ranks=[4, 5, 6, 7, 8, 9],
    n_initializations=1,
    max_iterations=100
)

Running multi-sample NMF on 72 samples.
Tested ranks: [4, 5, 6, 7, 8, 9]


Global NMF Progress:   0%|          | 0/43200 [00:00<?, ?it/s]

/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/nmf.py:104: UserWarning: Number of rows/columns is smaller than rank; results may be unstable.
  warnings.warn(
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/nmf.py:104: UserWarning: Number of rows/columns is smaller than rank; results may be unstable.
  warnings.warn(
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/nmf.py:104: UserWarning: Number of rows/columns is smaller than rank; results may be unstable.
  warnings.warn(
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/nmf.py:104: UserWarning: Number of rows/columns is smaller than rank; results may be unstable.
  warnings.warn(
/net/bq-storage/ag-cherrmann/bq_cinac/pr

Multi-sample NMF complete.


/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/nmf.py:104: UserWarning: Number of rows/columns is smaller than rank; results may be unstable.
  warnings.warn(


In [6]:
results_mp = kaleidocell.derive_nmf_metaprograms(results_nmf)
print(results_mp["metrics"])

Total number of programs: 2808
Optimal number of meta-programs: 30
Clustering using hierarchical Ward linkage
Defining meta-program consensus
Computing meta-program metrics
Meta-program derivation complete.
      sampleCoverage  silhouette  meanSimilarity  nPrograms  nGenes
MP1         0.013889    0.406116           0.449         21     328
MP2         0.013889    0.367070           0.402         18     167
MP3         0.013889    0.156387           0.272         29     325
MP4         0.013889    0.422055           0.466         19     281
MP5         0.013889    0.420846           0.478         18     529
MP6         0.013889    0.412746           0.471         18     524
MP7         0.013889    0.385052           0.449         18     330
MP8         0.027778    0.312601           0.360         26     488
MP9         0.069444    0.139228           0.204         42     487
MP10        0.013889    0.339580           0.391         19     451
MP11        0.166667    0.024789           0.

In [7]:
mp_scores = kaleidocell.compute_mp_scores(results_mp, adata)

Computing module scores per MP.
Finished computing MP scores.


In [8]:
path = kaleidocell.get_html(
    results_mp,
    adata,
    mp_scores=mp_scores,
    obs=["stage", "donor_id", "disease"],
    output_path="/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/02_NMF_immune_subsets/02_nmf_Neutrophil"
)
print(f"Report written to: {path}")

Rendering heatmap…
Rendering UMAP scores…
Building metrics table…
Running GSEA — GO Biological Process…
Ensembl gene IDs detected — translating to HGNC using bundled table.
Building gene table…
Rendering violin plots for 'stage'…
Plotting MP distributions across 'stage'.


/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Pas

Finished plotting MP distributions.
Rendering violin plots for 'donor_id'…
Plotting MP distributions across 'donor_id'.


/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Pas

Finished plotting MP distributions.
Rendering violin plots for 'disease'…
Plotting MP distributions across 'disease'.


/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Pas

Finished plotting MP distributions.

Files written to /net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/02_NMF_immune_subsets/02_nmf_Neutrophil/
  results.html
  gsea_GO_Biological_Process.csv
  genes.csv
Report written to: /net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/02_NMF_immune_subsets/02_nmf_Neutrophil/results.html
